
# DNTC One-Shot Auto OCR Pipeline — Kaggle + Google Drive

Notebook này chạy theo hướng **một phát ra final**, không cần review tay ở giữa:

1. Download PDF từ Google Drive.
2. Extract text layer bằng PyMuPDF.
3. Tự phát hiện page/line nào có text layer hỏng.
4. OCR ảnh bằng Tesseract Vietnamese cho page/line hỏng.
5. Fuse giữa `text_layer` và `image_ocr`, không replace mù.
6. Domain post-correction cho lỗi OCR lặp lại của *Đại Nam nhất thống chí*.
7. Reflow paragraph, tách câu, xuất CSV/JSONL/TXT/ZIP.

Điểm quan trọng: notebook này **không dùng PaddleOCR mặc định** vì trên Kaggle dễ lỗi dependency/model và với tiếng Việt scan cũ thường mất dấu. Tesseract + text-layer + rule/domain correction ổn định hơn cho one-shot.


In [ ]:

# ============================================================
# 0. CONFIG — chỉnh ở đây rồi Run All
# ============================================================
from pathlib import Path
import os

# Public Google Drive folder/file URL.
# Có thể thay bằng folder Drive của bạn. Notebook sẽ tải toàn bộ PDF trong folder.
DRIVE_URLS = [
    "https://drive.google.com/drive/folders/1QZzyaozPRLcm5Y2nmUUbigyVFnsX_tkW",
]

# Nếu không muốn download Drive, để [] và notebook sẽ đọc PDF từ /kaggle/input.
ALLOW_KAGGLE_INPUT_FALLBACK = True

OUTPUT_DIR = Path("/kaggle/working/dntc_one_shot_auto") if Path("/kaggle/working").exists() else Path("./dntc_one_shot_auto")
RAW_DIR = OUTPUT_DIR / "raw_drive"
DATA_DIR = OUTPUT_DIR / "data"
AUDIT_DIR = OUTPUT_DIR / "audit"
TEXT_DIR = OUTPUT_DIR / "final" / "texts"
PKG_DIR = OUTPUT_DIR / "packages"
CACHE_DIR = OUTPUT_DIR / "cache"
CROP_DIR = OUTPUT_DIR / "audit" / "crops"

# Runtime controls
FORCE_REDOWNLOAD = False
FORCE_REEXTRACT = True
MAX_PAGES_PER_PDF = None       # None = all pages
SKIP_VISUALLY_BLANK_PAGES = True

# OCR controls
ENABLE_TESSERACT = True
OCR_LANG = "vie+eng"
PAGE_OCR_DPI = 260             # 220-300. Higher = better but slower.
LINE_OCR_ZOOM = 4.5
LINE_OCR_PAD_PT = 8.0
LINE_OCR_MAX_TOTAL = 3500      # line-level OCR budget. Page-level OCR has no cap.

# Text-layer quality thresholds
BAD_PAGE_CONTROL_RATIO = 0.08
BAD_PAGE_MIN_LETTERS = 40
BAD_LINE_SCORE_THRESHOLD = 5

# Sentence/paragraph
MIN_SENT_CHARS = 2

# Export
WRITE_XLSX_PREVIEW = False     # CSV is faster and safer on Kaggle.
ZIP_OUTPUT = True

for d in [OUTPUT_DIR, RAW_DIR, DATA_DIR, AUDIT_DIR, TEXT_DIR, PKG_DIR, CACHE_DIR, CROP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("RAW_DIR:", RAW_DIR)


In [ ]:

# ============================================================
# 1. Install dependencies — Run All friendly, no restart required
# ============================================================
import sys, subprocess, shutil, importlib.util, os


def pip_install(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *pkgs]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)

# Python packages
for mod, pkgs in {
    "fitz": ["pymupdf"],
    "pandas": ["pandas"],
    "PIL": ["pillow"],
    "tqdm": ["tqdm"],
    "gdown": ["gdown"],
    "rapidfuzz": ["rapidfuzz"],
    "openpyxl": ["openpyxl"],
    "pytesseract": ["pytesseract"],
}.items():
    if importlib.util.find_spec(mod) is None:
        pip_install(*pkgs)

# Tesseract binary + Vietnamese traineddata.
# Kaggle normally allows apt-get when Internet is ON.
if ENABLE_TESSERACT and shutil.which("tesseract") is None:
    print("Installing tesseract system packages...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr", "tesseract-ocr-vie", "tesseract-ocr-eng"], check=False)

print("tesseract:", shutil.which("tesseract"))


In [ ]:

# ============================================================
# 2. Download / discover PDFs
# ============================================================
import os, re, shutil, subprocess, sys
from pathlib import Path
import gdown


def is_drive_url(url: str) -> bool:
    return isinstance(url, str) and "drive.google.com" in url


def download_drive_url(url: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    if not url or "PASTE" in url:
        return
    print("Downloading from Drive:", url)
    try:
        if "/folders/" in url:
            gdown.download_folder(url, output=str(out_dir), quiet=False, use_cookies=False, remaining_ok=True)
        else:
            gdown.download(url, output=str(out_dir), quiet=False, fuzzy=True)
    except TypeError:
        # Older gdown may not support remaining_ok/fuzzy
        try:
            if "/folders/" in url:
                subprocess.run([sys.executable, "-m", "gdown", "--folder", url, "-O", str(out_dir)], check=False)
            else:
                subprocess.run([sys.executable, "-m", "gdown", url, "-O", str(out_dir)], check=False)
        except Exception as e:
            print("Drive download fallback failed:", e)
    except Exception as e:
        print("Drive download failed:", repr(e))


if FORCE_REDOWNLOAD and RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_URLS:
    existing = list(RAW_DIR.rglob("*.pdf"))
    if FORCE_REDOWNLOAD or not existing:
        for url in DRIVE_URLS:
            download_drive_url(url, RAW_DIR)

pdf_candidates = []
pdf_candidates += list(RAW_DIR.rglob("*.pdf"))
if ALLOW_KAGGLE_INPUT_FALLBACK and Path("/kaggle/input").exists():
    pdf_candidates += list(Path("/kaggle/input").rglob("*.pdf"))

# Dedupe by resolved path and file size/name.
seen = set()
pdf_paths = []
for p in sorted(pdf_candidates):
    try:
        key = (p.name.lower(), p.stat().st_size)
        if key in seen:
            continue
        seen.add(key)
        pdf_paths.append(p)
    except Exception:
        pass

if not pdf_paths:
    raise FileNotFoundError("No PDFs found. Check DRIVE_URLS, Internet=On, or /kaggle/input.")

print("PDF count:", len(pdf_paths))
for p in pdf_paths[:50]:
    print("-", p)
if len(pdf_paths) > 50:
    print("...", len(pdf_paths) - 50, "more")


In [ ]:

# ============================================================
# 3. Core text normalization, scoring, and domain correction
# ============================================================
import re, unicodedata, math, json, hashlib, zipfile
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter, defaultdict

import pandas as pd
import fitz
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
from tqdm.auto import tqdm
from rapidfuzz import fuzz

try:
    import pytesseract
    from pytesseract import Output
except Exception:
    pytesseract = None
    Output = None

VIET_CHARS = set(
    "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩị"
    "óòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ"
    "ĂÂĐÊÔƠƯÁÀẢÃẠẮẰẲẴẶẤẦẨẪẬÉÈẺẼẸẾỀỂỄỆÍÌỈĨỊ"
    "ÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ"
)
CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\ufffe\uffff\u0001]")
WEIRD_RE = re.compile(r"[^\w\sÀ-ỹ.,;:!?(){}\[\]\"'“”‘’/\-–—%°+&]", re.UNICODE)
NON_VIET_LATIN_RE = re.compile(r"[Α-ωА-яЁё]")
LETTERS_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]")


def norm_space(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u00a0", " ")
    s = s.replace("\r", " ").replace("\n", " ")
    s = s.replace("￾", " ")
    s = CONTROL_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def digit_tokens(s):
    return re.findall(r"\d+(?:[.,]\d+)?", norm_space(s))


def weird_count(s):
    s = norm_space(s)
    return len(WEIRD_RE.findall(s)) + 3 * len(NON_VIET_LATIN_RE.findall(s))


def accent_count(s):
    return sum(1 for ch in norm_space(s) if ch in VIET_CHARS)


def control_count(s):
    return len(CONTROL_RE.findall(str(s or "")))


def similarity(a, b):
    a, b = norm_space(a), norm_space(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


COMMON_VI_WORDS = set("""
của và là có không trong ngoài năm đời phủ huyện tỉnh châu xã thôn sông núi biển đông tây nam bắc
thành đặt đổi thuộc giáp cách dặm phía quyền quyển đại nam nhất thống chí kinh sư phần dã dựng đặt diễn cách
hình thế khí hậu phong tục thành trì trường học hộ khẩu thuế ruộng núi sông cổ tích từ miếu đền chùa
minh mệnh gia long tự đức thiệu trị đồng khánh duy tân hiến tông duệ tông thế tổ thánh tông
""".split())

DOMAIN_PHRASES = [
    "ĐẠI NAM NHẤT THỐNG CHÍ", "DỰNG ĐẶT VÀ DIÊN CÁCH", "PHẦN DÃ", "HÌNH THẾ", "KHÍ HẬU", "PHONG TỤC",
    "THÀNH TRÌ", "TRƯỜNG HỌC", "HỘ KHẨU", "THUẾ RUỘNG", "NÚI SÔNG", "Tự Đức", "Minh Mệnh", "Thiệu Trị",
    "Gia Long", "Hà Tiên", "Quảng Bình", "Bình Định", "Quảng Yên", "Kinh sư", "Cao Mên", "Chiêm Thành",
]

# Rule list: conservative, mostly repeated OCR errors seen in the DNTC scans.
PHRASE_RULES = [
    # Titles / headings
    (r"\bDAI\s*-?\s*NAM\b", "ĐẠI NAM"),
    (r"\bĐAI\s*-?\s*NAM\b", "ĐẠI NAM"),
    (r"\bNH[ẤA]T\s+THONG\s+CH[ÍI]\b", "NHẤT THỐNG CHÍ"),
    (r"\bNHAT\s+THONG\s+CHI\b", "NHẤT THỐNG CHÍ"),
    (r"\bĐẠI\s+NAM\s+NHẬT\s+THỐNG\s+CHÍ\b", "ĐẠI NAM NHẤT THỐNG CHÍ"),
    (r"\bĐẠI\s+NAM\s+NHẤT\s+THONG\s+CHÍ\b", "ĐẠI NAM NHẤT THỐNG CHÍ"),
    (r"\bQUYEN\b", "QUYỂN"),
    (r"\bTINH\b", "TỈNH"),
    (r"\bKINH\s+SU\b", "KINH SƯ"),
    (r"\bPHAN\s+DA\b", "PHẦN DÃ"),
    (r"\bPHẦN\s+DA\b", "PHẦN DÃ"),
    (r"\bDUNG\s+[DP]AT\s+VA\s+DIEN\s+CACH\b", "DỰNG ĐẶT VÀ DIÊN CÁCH"),
    (r"\bDỰNG\s+ĐẶT\s+VA\s+DIỄN\s+CÁCH\b", "DỰNG ĐẶT VÀ DIÊN CÁCH"),
    (r"\bHINH\s+THE\b", "HÌNH THẾ"),
    (r"\bPHONG\s+TUC\b", "PHONG TỤC"),
    (r"\bTHUE\s+RUONG\b", "THUẾ RUỘNG"),
    (r"\bNUI\s+SONG\b", "NÚI SÔNG"),
    # Common names / dynastic terms
    (r"\bMinh\s+M[ée]nh\b", "Minh Mệnh"),
    (r"\bMinh\s+Mộệnh\b", "Minh Mệnh"),
    (r"\bThiệu\s+Tri\b", "Thiệu Trị"),
    (r"\bTự\s+Dức\b", "Tự Đức"),
    (r"\bDự\s+Đức\b", "Tự Đức"),
    (r"#ự\s+Đức", "Tự Đức"),
    (r"\bdoi\s+#?ự\s+Đức\b", "đời Tự Đức"),
    (r"\bdoi\s+H[ií]én\s+Tông\b", "đời Hiến Tông"),
    (r"\bH[ií]én\s+Tông\b", "Hiến Tông"),
    (r"\bGia\s+Du\s+Hoàng\b", "Gia Dụ Hoàng"),
    (r"\bCao\s+Hoàng\s+Đ[ếe]t\b", "Cao Hoàng Đế"),
    (r"\bNha\s+Tuy\b", "Nhà Tùy"),
    (r"\bNha\s+Đường\b", "Nhà Đường"),
    (r"\bNước\s+tả\b", "Nước ta"),
    # Place names / headings
    (r"\bTỈNH\s+HA\s+TIEN\b", "TỈNH HÀ TIÊN"),
    (r"\bTINH\s+HA\s+TIEN\b", "TỈNH HÀ TIÊN"),
    (r"\bTỈNH\s+QUANG\s+BINH\b", "TỈNH QUẢNG BÌNH"),
    (r"\bTINH\s+QUANG\s+BINH\b", "TỈNH QUẢNG BÌNH"),
    (r"\bTỈNH\s+BINH\s+DINH\b", "TỈNH BÌNH ĐỊNH"),
    (r"\bTINH\s+BINH\s+DINH\b", "TỈNH BÌNH ĐỊNH"),
    (r"\bTỈNH\s+QUANG\s+YEN\b", "TỈNH QUẢNG YÊN"),
    (r"\bTINH\s+QUANG\s+YEN\b", "TỈNH QUẢNG YÊN"),
    (r"\bHa\s+Tien\b", "Hà Tiên"),
    (r"\bQuang\s+Binh\b", "Quảng Bình"),
    (r"\bBinh\s+Dinh\b", "Bình Định"),
    (r"\bQuang\s+Yen\b", "Quảng Yên"),
    # Frequent OCR residues
    (r"\bth[eéế]\s+ky\b", "thế kỷ"),
    (r"\bth[eéế]\s+kỷ\b", "thế kỷ"),
    (r"\bbi€n\b", "biển"),
    (r"\bki€m\b", "kiêm"),
    (r"\bmi€u\b", "miếu"),
    (r"\bchi€m\b", "chiếm"),
    (r"\bcudp\b", "cướp"),
    (r"\bñăm\b", "năm"),
    (r"\bNii\b", "Núi"),
    (r"\bphd\b", "phủ"),
    (r"\bphd\s+Tĩnh\b", "phủ Tĩnh"),
    (r"\bhuyénhién\b", "huyện hiện"),
    (r"\bChiêmh\s+Thanh\b", "Chiêm Thành"),
    (r"\bChiệm\s+Thành\b", "Chiêm Thành"),
    (r"\bQuang\s+Todn\b", "Quang Toản"),
    (r"\bNguyÊn\s+Văn\s+Hud\b", "Nguyễn Văn Huệ"),
    (r"\bThuy\s+Xd\b", "Thủy Xá"),
    (r"\bséng\b", "sông"),
    (r"\bsin\b", "sản"),
    (r"\btrim\b", "trầm"),
    (r"\bday\s+thanh\b", "đây thành"),
]


def apply_domain_rules(s):
    s0 = norm_space(s)
    s = s0
    for pat, repl in PHRASE_RULES:
        s = re.sub(pat, repl, s, flags=re.IGNORECASE)
    # Fix spaced punctuation / digit-letter glue.
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,.;:!?])(\S)", r"\1 \2", s)
    s = re.sub(r"(\d)\s+([,.])\s+(\d)", r"\1\2\3", s)
    s = re.sub(r"(\d)([A-Za-zÀ-ỹĐđ])", r"\1 \2", s)
    s = re.sub(r"([A-Za-zÀ-ỹĐđ])(?=\d)", r"\1 ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def text_score(s):
    s = norm_space(s)
    if not s:
        return -999.0
    letters = LETTERS_RE.findall(s)
    words = re.findall(r"[A-Za-zÀ-ỹĐđ]+", s.lower())
    vi_word_hits = sum(1 for w in words if w in COMMON_VI_WORDS)
    phrase_hits = sum(1 for p in DOMAIN_PHRASES if p.lower() in s.lower())
    acc = accent_count(s)
    weird = weird_count(s)
    ctrl = control_count(s)
    digit_letter_glue = len(re.findall(r"[A-Za-zÀ-ỹĐđ]\d|\d[A-Za-zÀ-ỹĐđ]", s))
    ascii_ratio_pen = 0.0
    if len(letters) >= 30:
        ascii_letters = sum(1 for ch in letters if ord(ch) < 128)
        acc_ratio = acc / max(1, len(letters))
        if ascii_letters / max(1, len(letters)) > 0.85 and acc_ratio < 0.035:
            ascii_ratio_pen = 8.0
    return (
        len(s) * 0.01
        + acc * 0.9
        + vi_word_hits * 1.2
        + phrase_hits * 4.0
        - weird * 5.0
        - ctrl * 6.0
        - digit_letter_glue * 2.5
        - ascii_ratio_pen
    )

BAD_PATTERNS = re.compile("|".join([
    r"[€#ñ￾\u0001]",
    r"\bth[eéế]\s+k[yỷ]\b",
    r"\bNHẤT\s+THONG\b",
    r"\bDUNG\s+[DP]AT\b",
    r"\bPHAN\s+DA\b",
    r"\bNha\s+Tuy\b",
    r"\bNước\s+tả\b",
    r"\bcudp\b",
    r"\bchi€m\b|\bbi€n\b|\bki€m\b|\bmi€u\b",
    r"\bhuyénhién\b",
    r"[A-Za-zÀ-ỹĐđ]\d|\d[A-Za-zÀ-ỹĐđ]",
]), re.IGNORECASE)


def suspicious_score(s):
    s = norm_space(s)
    score = 0
    if not s:
        return 0
    if BAD_PATTERNS.search(s): score += 6
    score += min(10, weird_count(s) * 2)
    score += min(8, control_count(s) * 2)
    if re.search(r"\b[A-Z]{2,}[a-zà-ỹ]+", s): score += 2
    letters = LETTERS_RE.findall(s)
    if len(letters) >= 30:
        acc_ratio = accent_count(s) / max(1, len(letters))
        if acc_ratio < 0.025 and re.search(r"\b(tinh|huyen|phu|chau|song|nui|bien|thong|nhat|kinh)\b", s, re.I):
            score += 5
    if len(s) > 260: score += 2
    return score


def choose_best_text(base, ocr, reason_prefix=""):
    base0 = norm_space(base)
    ocr0 = norm_space(ocr)
    base1 = apply_domain_rules(base0)
    ocr1 = apply_domain_rules(ocr0)
    if not ocr1:
        return base1, f"{reason_prefix}keep_base:no_ocr", False
    # Protect digits and avoid truncation.
    bd = digit_tokens(base1)
    od = digit_tokens(ocr1)
    if bd:
        kept = sum(1 for d in bd if d in od)
        if kept / max(1, len(bd)) < 0.70:
            return base1, f"{reason_prefix}keep_base:ocr_lost_digits", False
    if len(ocr1) < 0.55 * max(1, len(base1)):
        return base1, f"{reason_prefix}keep_base:ocr_too_short", False
    if NON_VIET_LATIN_RE.search(ocr1):
        return base1, f"{reason_prefix}keep_base:ocr_non_viet_latin", False
    base_score = text_score(base1)
    ocr_score = text_score(ocr1)
    sim = similarity(base1, ocr1)
    base_bad = suspicious_score(base1)
    ocr_bad = suspicious_score(ocr1)
    # Accept only when OCR is meaningfully cleaner and related.
    if (ocr_score >= base_score + 5.0 and ocr_bad <= max(1, base_bad - 2) and sim >= 0.30):
        return ocr1, f"{reason_prefix}accept_ocr:score {base_score:.1f}->{ocr_score:.1f}; suspicious {base_bad}->{ocr_bad}; sim={sim:.2f}", True
    if base_bad >= 8 and ocr_bad <= 2 and ocr_score >= base_score and sim >= 0.25:
        return ocr1, f"{reason_prefix}accept_ocr:fixed_bad_pattern; sim={sim:.2f}", True
    return base1, f"{reason_prefix}keep_base:score {base_score:.1f}->{ocr_score:.1f}; suspicious {base_bad}->{ocr_bad}; sim={sim:.2f}", False


def page_text_quality(text):
    text = norm_space(text)
    letters = LETTERS_RE.findall(text)
    ctrl = control_count(text)
    total = max(1, len(text))
    weird = weird_count(text)
    return {
        "letters": len(letters),
        "controls": ctrl,
        "control_ratio": ctrl / total,
        "weird": weird,
        "score": text_score(text),
        "is_bad": (len(letters) < BAD_PAGE_MIN_LETTERS) or (ctrl / total > BAD_PAGE_CONTROL_RATIO) or (weird > 25),
    }


def is_noise_line(s):
    s = norm_space(s)
    if not s:
        return True
    if re.fullmatch(r"\d{1,4}", s):
        return True
    if re.search(r"Digitized by\s+Google", s, re.I):
        return True
    if re.fullmatch(r"[.\-–—_\s]+", s):
        return True
    # Many control glyphs or repeated junk.
    if control_count(s) > max(3, len(s) * 0.25):
        return True
    return False


def is_heading(s):
    s = norm_space(s)
    if not s or len(s) > 90:
        return False
    su = s.upper()
    heading_terms = ["PHẦN DÃ", "DỰNG ĐẶT", "HÌNH THẾ", "KHÍ HẬU", "PHONG TỤC", "THÀNH TRÌ", "TRƯỜNG HỌC", "HỘ KHẨU", "THUẾ RUỘNG", "NÚI SÔNG", "ĐẠI NAM", "QUYỂN", "TỈNH"]
    if any(t in su for t in heading_terms):
        return True
    letters = LETTERS_RE.findall(s)
    if letters:
        upperish = sum(1 for ch in letters if ch.upper() == ch) / len(letters)
        if upperish > 0.75 and len(s) < 70:
            return True
    return False


In [ ]:

# ============================================================
# 4. OCR wrappers and PDF line extraction
# ============================================================
import numpy as np


def pil_preprocess(img: Image.Image, mode="page") -> Image.Image:
    img = img.convert("RGB")
    gray = ImageOps.grayscale(img)
    gray = ImageOps.autocontrast(gray)
    gray = ImageEnhance.Contrast(gray).enhance(1.35 if mode == "page" else 1.65)
    gray = ImageEnhance.Sharpness(gray).enhance(1.25 if mode == "page" else 1.55)
    if mode == "line":
        gray = gray.filter(ImageFilter.MedianFilter(size=3))
    return gray.convert("RGB")


def render_page_to_pil(page, dpi=PAGE_OCR_DPI):
    zoom = dpi / 72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return img, zoom


def render_rect_to_pil(page, rect, zoom=LINE_OCR_ZOOM, pad=LINE_OCR_PAD_PT):
    r = fitz.Rect(rect)
    r.x0 = max(page.rect.x0, r.x0 - pad)
    r.y0 = max(page.rect.y0, r.y0 - pad)
    r.x1 = min(page.rect.x1, r.x1 + pad)
    r.y1 = min(page.rect.y1, r.y1 + pad)
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), clip=r, alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return img


def tesseract_available():
    return ENABLE_TESSERACT and pytesseract is not None and shutil.which("tesseract") is not None


def tesseract_image_to_text(img, psm=6):
    if not tesseract_available():
        return "", 0.0
    try:
        config = f"--oem 1 --psm {psm}"
        txt = pytesseract.image_to_string(img, lang=OCR_LANG, config=config)
        txt = norm_space(txt)
        return txt, (0.55 if txt else 0.0)
    except Exception as e:
        return "", 0.0


def tesseract_page_to_lines(page, page_idx, work_id, pdf_path):
    """Full-page OCR fallback for pages with bad/empty text layer."""
    if not tesseract_available():
        return []
    img, zoom = render_page_to_pil(page, dpi=PAGE_OCR_DPI)
    img = pil_preprocess(img, mode="page")
    try:
        df = pytesseract.image_to_data(img, lang=OCR_LANG, config="--oem 1 --psm 6", output_type=Output.DATAFRAME)
    except Exception as e:
        return []
    if df is None or len(df) == 0:
        return []
    df = df.dropna(subset=["text"])
    if df.empty:
        return []
    df["text"] = df["text"].map(norm_space)
    df = df[df["text"].str.len() > 0]
    # Filter low confidence but keep if text has letters; cover pages often have weird confidence.
    if "conf" in df.columns:
        df["conf_num"] = pd.to_numeric(df["conf"], errors="coerce").fillna(-1)
        df = df[(df["conf_num"] >= 0) | (df["text"].str.contains(r"[A-Za-zÀ-ỹĐđ]", regex=True))]
    lines = []
    group_cols = [c for c in ["block_num", "par_num", "line_num"] if c in df.columns]
    if not group_cols:
        group_cols = ["level"] if "level" in df.columns else []
    if not group_cols:
        return []
    for key, g in df.groupby(group_cols, sort=True):
        words = [norm_space(x) for x in g["text"].tolist() if norm_space(x)]
        text = norm_space(" ".join(words))
        if is_noise_line(text):
            continue
        left = float(g["left"].min()) / zoom
        top = float(g["top"].min()) / zoom
        right = float((g["left"] + g["width"]).max()) / zoom
        bottom = float((g["top"] + g["height"]).max()) / zoom
        conf = float(pd.to_numeric(g.get("conf", pd.Series([-1])), errors="coerce").fillna(-1).mean()) if "conf" in g.columns else -1
        lines.append({
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_idx + 1,
            "source": "tesseract_page", "bbox": [left, top, right, bottom], "raw_text": text,
            "ocr_conf": conf,
        })
    return lines


def pymupdf_page_to_lines(page, page_idx, work_id, pdf_path):
    out = []
    try:
        data = page.get_text("dict")
    except Exception:
        return out
    for b_i, block in enumerate(data.get("blocks", [])):
        if block.get("type") != 0:
            continue
        for l_i, line in enumerate(block.get("lines", [])):
            spans = line.get("spans", [])
            text = norm_space(" ".join([sp.get("text", "") for sp in spans]))
            if is_noise_line(text):
                continue
            bbox = line.get("bbox", block.get("bbox", None))
            if not bbox:
                continue
            out.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_idx + 1,
                "source": "pdf_text_layer", "bbox": [float(x) for x in bbox], "raw_text": text,
                "block_id": b_i, "line_id": l_i, "ocr_conf": None,
            })
    out.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return out


def line_crop_ocr(page, line_record):
    rect = fitz.Rect(line_record["bbox"])
    img = render_rect_to_pil(page, rect, zoom=LINE_OCR_ZOOM, pad=LINE_OCR_PAD_PT)
    variants = [pil_preprocess(img, mode="line"), img]
    best_txt, best_score = "", -999
    for im in variants:
        txt, _ = tesseract_image_to_text(im, psm=7)
        sc = text_score(txt)
        if sc > best_score:
            best_txt, best_score = txt, sc
    return best_txt


def improve_line_with_ocr(doc, line_record, budget_state):
    base = line_record.get("raw_text", "")
    base_fixed = apply_domain_rules(base)
    if suspicious_score(base_fixed) < BAD_LINE_SCORE_THRESHOLD:
        return base_fixed, "rules_only", False, ""
    if budget_state["used"] >= LINE_OCR_MAX_TOTAL or not tesseract_available():
        return base_fixed, "no_line_ocr_budget_or_engine", False, ""
    try:
        page = doc[line_record["page_idx"]]
        ocr_txt = line_crop_ocr(page, line_record)
        budget_state["used"] += 1
        chosen, reason, accepted = choose_best_text(base_fixed, ocr_txt, reason_prefix="line_ocr:")
        return chosen, reason, accepted, ocr_txt
    except Exception as e:
        return base_fixed, f"line_ocr_failed:{type(e).__name__}", False, ""


In [ ]:

# ============================================================
# 5. Paragraph reflow and sentence splitting
# ============================================================

def median_line_height(lines):
    hs = []
    for r in lines:
        try:
            b = r["bbox"]
            hs.append(max(1.0, float(b[3]) - float(b[1])))
        except Exception:
            pass
    return float(np.median(hs)) if hs else 12.0


def should_start_new_para(prev, cur, med_h):
    pt, ct = prev["text"], cur["text"]
    if is_heading(ct) or is_heading(pt):
        return True
    try:
        gap = cur["bbox"][1] - prev["bbox"][3]
        indent_delta = cur["bbox"][0] - prev["bbox"][0]
        if gap > med_h * 0.85:
            return True
        if indent_delta > med_h * 1.5 and len(pt) > 20:
            return True
    except Exception:
        pass
    if re.match(r"^\(?\d+\)|^\d+[.)]", ct):
        return True
    return False


def join_lines_text(lines):
    parts = []
    for r in lines:
        t = norm_space(r["text"])
        if not t:
            continue
        if not parts:
            parts.append(t)
            continue
        prev = parts[-1]
        if prev.endswith("-") and not re.search(r"\s-$", prev):
            parts[-1] = prev[:-1] + t
        elif re.search(r"[\-–—]$", prev) and len(prev) < 35:
            parts.append(t)
        else:
            parts.append(t)
    text = " ".join(parts)
    text = re.sub(r"\s+", " ", text).strip()
    return apply_domain_rules(text)


def reflow_page_lines(lines):
    if not lines:
        return []
    lines = [dict(r) for r in lines if not is_noise_line(r.get("text") or r.get("raw_text"))]
    for r in lines:
        r["text"] = apply_domain_rules(r.get("text") or r.get("raw_text", ""))
    med_h = median_line_height(lines)
    paras = []
    cur = []
    for r in lines:
        if not cur:
            cur = [r]
            continue
        if should_start_new_para(cur[-1], r, med_h):
            paras.append(cur)
            cur = [r]
        else:
            cur.append(r)
    if cur:
        paras.append(cur)
    out = []
    for p_i, group in enumerate(paras):
        text = join_lines_text(group)
        if not text:
            continue
        x0 = min(g["bbox"][0] for g in group)
        y0 = min(g["bbox"][1] for g in group)
        x1 = max(g["bbox"][2] for g in group)
        y1 = max(g["bbox"][3] for g in group)
        typ = "heading" if len(group) <= 2 and is_heading(text) else "paragraph"
        out.append({
            "paragraph_index_in_page": p_i,
            "type": typ,
            "text": text,
            "bbox": [x0, y0, x1, y1],
            "line_count": len(group),
            "sources": ";".join(sorted(set(g.get("source", "") for g in group))),
            "ocr_accepted_count": sum(1 for g in group if g.get("ocr_accepted")),
            "correction_notes": " | ".join([g.get("correction_note", "") for g in group if g.get("correction_note")])[:1000],
        })
    return out


def protect_abbrev(text):
    repl = {
        "v.v..": "v§v§.",
        "v.v.": "v§v§",
        "tr.C.N.": "tr§C§N§",
        "T.P.": "T§P§",
        "A.69": "A§69",
        "HV.140": "HV§140",
        "HV.38": "HV§38",
    }
    for k, v in repl.items():
        text = text.replace(k, v)
    return text, repl


def unprotect_abbrev(text, repl):
    inv = {v: k for k, v in repl.items()}
    for k, v in inv.items():
        text = text.replace(k, v)
    text = text.replace("§", ".")
    return text


def split_sentences(text, typ="paragraph"):
    text = apply_domain_rules(text)
    if typ == "heading":
        return [text]
    protected, repl = protect_abbrev(text)
    # Split after sentence punctuation. Do not split at semicolon by default; historical prose uses semicolons heavily.
    parts = re.split(r"(?<=[.!?…])\s+(?=[A-ZÀ-ỸĐ0-9\(\"“])", protected)
    out = []
    for p in parts:
        p = unprotect_abbrev(norm_space(p), repl)
        if not p:
            continue
        # Merge tiny fragments into previous.
        if out and len(p) < 12 and not is_heading(p):
            out[-1] = norm_space(out[-1] + " " + p)
        else:
            out.append(p)
    return [s for s in out if len(s) >= MIN_SENT_CHARS]


In [ ]:

# ============================================================
# 6. Process all PDFs
# ============================================================
import time

all_lines = []
all_paragraphs = []
all_sentences = []
page_reports = []
line_audit = []
ocr_budget = {"used": 0}


def make_work_id(pdf_path, existing_ids):
    base = re.sub(r"[^A-Za-z0-9_\-]+", "_", pdf_path.stem).strip("_") or "pdf"
    wid = base
    k = 2
    while wid in existing_ids:
        wid = f"{base}_{k}"
        k += 1
    existing_ids.add(wid)
    return wid

work_ids = set()
start_time = time.time()

for pdf_i, pdf_path in enumerate(pdf_paths, 1):
    work_id = make_work_id(pdf_path, work_ids)
    print(f"\n[{pdf_i}/{len(pdf_paths)}] {work_id}: {pdf_path}")
    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        print("Cannot open PDF:", e)
        continue
    n_pages = len(doc)
    page_limit = n_pages if MAX_PAGES_PER_PDF is None else min(n_pages, MAX_PAGES_PER_PDF)
    print("pages:", n_pages, "processing:", page_limit)
    for page_idx in tqdm(range(page_limit), desc=work_id):
        page = doc[page_idx]
        text_layer_text = norm_space(page.get_text("text"))
        q = page_text_quality(text_layer_text)
        use_page_ocr = q["is_bad"]
        if use_page_ocr and tesseract_available():
            lines = tesseract_page_to_lines(page, page_idx, work_id, pdf_path)
            page_source = "tesseract_page" if lines else "bad_text_layer_empty_ocr"
            # If Tesseract fails but text layer has some text, fallback to text layer.
            if not lines and q["letters"] >= 10:
                lines = pymupdf_page_to_lines(page, page_idx, work_id, pdf_path)
                page_source = "pdf_text_layer_fallback"
        else:
            lines = pymupdf_page_to_lines(page, page_idx, work_id, pdf_path)
            page_source = "pdf_text_layer"
        # Skip visually blank/no-text pages.
        if not lines:
            page_reports.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_number": page_idx + 1,
                "source": page_source, **q, "line_count": 0, "skipped": True,
            })
            continue
        improved_lines = []
        for li, r in enumerate(lines):
            raw = r.get("raw_text", "")
            fixed, note, accepted, ocr_txt = improve_line_with_ocr(doc, r, ocr_budget) if r.get("source") == "pdf_text_layer" else (apply_domain_rules(raw), "page_ocr_or_rules", False, "")
            rr = dict(r)
            rr["line_global_id"] = f"{work_id}_p{page_idx+1:04d}_l{li+1:03d}"
            rr["text"] = fixed
            rr["raw_text"] = raw
            rr["ocr_line_text"] = ocr_txt
            rr["ocr_accepted"] = bool(accepted)
            rr["correction_note"] = note
            rr["suspicious_before"] = suspicious_score(raw)
            rr["suspicious_after"] = suspicious_score(fixed)
            improved_lines.append(rr)
            all_lines.append(rr)
            if note != "rules_only" or rr["suspicious_after"] >= BAD_LINE_SCORE_THRESHOLD:
                line_audit.append({
                    "work_id": work_id, "page_number": page_idx + 1, "line_id": rr["line_global_id"],
                    "source": rr.get("source"), "raw_text": raw, "final_line_text": fixed,
                    "ocr_line_text": ocr_txt, "ocr_accepted": accepted, "note": note,
                    "suspicious_before": rr["suspicious_before"], "suspicious_after": rr["suspicious_after"],
                    "bbox": json.dumps(rr.get("bbox", []), ensure_ascii=False),
                })
        paras = reflow_page_lines(improved_lines)
        for para in paras:
            paragraph_id = f"{work_id}_p{page_idx+1:04d}_para{para['paragraph_index_in_page']+1:03d}"
            prow = {
                "paragraph_id": paragraph_id,
                "work_id": work_id,
                "pdf_path": str(pdf_path),
                "page_idx": page_idx,
                "page_number": page_idx + 1,
                **para,
                "bbox": json.dumps(para["bbox"], ensure_ascii=False),
                "quality_score": suspicious_score(para["text"]),
            }
            all_paragraphs.append(prow)
            sents = split_sentences(para["text"], para["type"])
            for si, sent in enumerate(sents):
                sid = f"{work_id}_s{len(all_sentences)+1:07d}"
                all_sentences.append({
                    "sent_id": sid,
                    "paragraph_id": paragraph_id,
                    "work_id": work_id,
                    "pdf_path": str(pdf_path),
                    "page_idx": page_idx,
                    "page_number": page_idx + 1,
                    "sentence_index_in_paragraph": si,
                    "type": para["type"] if si == 0 else "sentence",
                    "text": sent,
                    "bbox": prow["bbox"],
                    "sources": para["sources"],
                    "quality_score": suspicious_score(sent),
                    "text_score": text_score(sent),
                    "ocr_accepted_count": para["ocr_accepted_count"],
                    "correction_notes": para["correction_notes"],
                })
        page_reports.append({
            "work_id": work_id, "pdf_path": str(pdf_path), "page_number": page_idx + 1,
            "source": page_source, **q, "line_count": len(lines), "paragraph_count": len(paras), "skipped": False,
        })
    doc.close()

elapsed = time.time() - start_time
print("\nDone.")
print("Lines:", len(all_lines), "Paragraphs:", len(all_paragraphs), "Sentences:", len(all_sentences))
print("Line OCR used:", ocr_budget["used"], "/", LINE_OCR_MAX_TOTAL)
print("Elapsed minutes:", round(elapsed/60, 2))


In [ ]:

# ============================================================
# 7. Export final files + audits
# ============================================================
import pandas as pd, json, zipfile, os
from pathlib import Path

lines_df = pd.DataFrame(all_lines)
paras_df = pd.DataFrame(all_paragraphs)
sents_df = pd.DataFrame(all_sentences)
pages_df = pd.DataFrame(page_reports)
audit_df = pd.DataFrame(line_audit)

# Sort consistently
if not sents_df.empty:
    sents_df = sents_df.sort_values(["work_id", "page_number", "paragraph_id", "sentence_index_in_paragraph"]).reset_index(drop=True)

DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

lines_csv = DATA_DIR / "lines_fused.csv"
paras_csv = DATA_DIR / "paragraphs_fused.csv"
sents_csv = OUTPUT_DIR / "final" / "final_sentences_auto.csv"
sents_jsonl = OUTPUT_DIR / "final" / "final_sentences_auto.jsonl"
pages_csv = AUDIT_DIR / "page_quality_report.csv"
audit_csv = AUDIT_DIR / "line_correction_audit.csv"
remaining_csv = AUDIT_DIR / "remaining_suspicious_sentences.csv"
summary_csv = AUDIT_DIR / "summary.csv"

(OUTPUT_DIR / "final").mkdir(parents=True, exist_ok=True)

lines_df.to_csv(lines_csv, index=False, encoding="utf-8-sig")
paras_df.to_csv(paras_csv, index=False, encoding="utf-8-sig")
sents_df.to_csv(sents_csv, index=False, encoding="utf-8-sig")
pages_df.to_csv(pages_csv, index=False, encoding="utf-8-sig")
audit_df.to_csv(audit_csv, index=False, encoding="utf-8-sig")

with open(sents_jsonl, "w", encoding="utf-8") as f:
    for rec in sents_df.to_dict("records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Remaining suspicious: this is an audit queue, not the final error rate.
remaining_df = sents_df[sents_df["quality_score"].fillna(0).astype(float) >= BAD_LINE_SCORE_THRESHOLD].copy() if not sents_df.empty else pd.DataFrame()
remaining_df.to_csv(remaining_csv, index=False, encoding="utf-8-sig")

# Per-work final txt
for work_id, g in sents_df.groupby("work_id", dropna=False):
    out = TEXT_DIR / f"{work_id}_final_auto.txt"
    parts = []
    cur_page = None
    for _, r in g.iterrows():
        if r["page_number"] != cur_page:
            cur_page = r["page_number"]
            parts.append(f"\n\n[Page {cur_page}]\n")
        parts.append(norm_space(r["text"]))
    out.write_text("\n".join([p for p in parts if norm_space(p)]), encoding="utf-8")

summary = pd.DataFrame([{
    "pdf_count": len(pdf_paths),
    "pages_total": len(pages_df),
    "pages_skipped": int(pages_df.get("skipped", pd.Series(dtype=bool)).sum()) if not pages_df.empty else 0,
    "lines": len(lines_df),
    "paragraphs": len(paras_df),
    "sentences": len(sents_df),
    "line_ocr_used": ocr_budget["used"],
    "remaining_suspicious_sentences": len(remaining_df),
    "tesseract_available": tesseract_available(),
    "output_dir": str(OUTPUT_DIR),
}])
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print(summary.to_string(index=False))
print("\nFinal CSV:", sents_csv)
print("Final JSONL:", sents_jsonl)
print("Final TXT dir:", TEXT_DIR)
print("Remaining suspicious:", remaining_csv)

if WRITE_XLSX_PREVIEW:
    try:
        preview_xlsx = AUDIT_DIR / "remaining_suspicious_preview.xlsx"
        remaining_df.head(5000).to_excel(preview_xlsx, index=False)
        print("XLSX preview:", preview_xlsx)
    except Exception as e:
        print("Could not write XLSX preview:", e)

if ZIP_OUTPUT:
    zip_path = PKG_DIR / "dntc_one_shot_auto_output.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in [lines_csv, paras_csv, sents_csv, sents_jsonl, pages_csv, audit_csv, remaining_csv, summary_csv]:
            if Path(p).exists():
                zf.write(p, arcname=str(Path(p).relative_to(OUTPUT_DIR)))
        for p in TEXT_DIR.glob("*.txt"):
            zf.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))
    print("ZIP:", zip_path)


In [ ]:

# ============================================================
# 8. Quick automatic quality view
# ============================================================
if not sents_df.empty:
    display_cols = ["work_id", "page_number", "quality_score", "text"]
    print("Top remaining suspicious examples:")
    display(sents_df.sort_values("quality_score", ascending=False)[display_cols].head(30))

    print("\nRandom sample:")
    display(sents_df.sample(min(20, len(sents_df)), random_state=42)[display_cols])

if not pages_df.empty:
    print("\nPage source distribution:")
    display(pages_df.groupby(["source", "skipped"]).size().reset_index(name="count"))
